# Notebook 05 — Physics-constrained multi-task learning for insertion-electrode properties

**Private execution notebook.** This notebook is part of the current Li–Na–K computational-materials workflow. It is not the public-repository version.

## Scientific purpose

This notebook compares matched multi-output models under the validation hierarchy established in Notebook 04:

1. **Unconstrained direct prediction** of linked electrode properties.
2. **Hard-derived prediction**, where energy and worst-case stability are reconstructed from predicted primitive quantities.
3. **Soft physics-constrained prediction**, where the training loss penalizes violations of known property relationships.

The physical relations tested are:

- `energy_grav = average_voltage × capacity_grav`
- `stability_worst = max(stability_charge, stability_discharge)`

The notebook does **not** use observed voltage or observed capacity as input features for clean energy prediction, and it does **not** use observed endpoint stabilities as input features for clean worst-stability prediction.

## Split-reuse policy

Notebook 04 did not persist a record-level split manifest. Therefore:

- leave-family, leave-chemical-system, and leave-working-ion folds are reconstructed from Notebook 04's recorded held-out groups;
- random and framework folds are regenerated using the same documented algorithms, seed, grouping field, fold counts, and valid-record universe;
- all methods inside Notebook 05 use exactly the same newly persisted record-level split manifest.

This avoids making an unsupported claim of record-identical framework folds across different NumPy/scikit-learn environments.

## Execution policy

This copy is configured for `FULL` mode. Restart the kernel and run all cells from the beginning.


In [ ]:
# ============================================================
# Cell 1 — User configuration
# ============================================================
# First execution: "SMOKE"
# Final execution after smoke test passes: "FULL"
RUN_MODE = "FULL"  # allowed: "SMOKE", "FULL"

# Full scientific run settings
FULL_PROTOCOLS = ["P1", "P2", "P3"]
FULL_SPLITS = [
    "random_split",
    "framework_groupkfold",
    "leave_family_out",
    "leave_chemical_system_out",
    "leave_working_ion_out",
]
INCLUDE_ALL_75_CHEMICAL_SYSTEM_HOLDOUTS = True

# Reproducibility and compute
RANDOM_STATE = 42
TORCH_NUM_THREADS = 4
BATCH_SIZE = 256
FULL_MAX_EPOCHS = 90
FULL_EARLY_STOPPING_PATIENCE = 12
SMOKE_MAX_EPOCHS = 25
SMOKE_EARLY_STOPPING_PATIENCE = 6
HIDDEN_DIMS = (64, 32)
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
INNER_VALIDATION_FRACTION = 0.12

# Dimensionless physics penalties. They are fixed before outer-test evaluation.
LAMBDA_ENERGY = 1.0
LAMBDA_STABILITY = 1.0
LAMBDA_NONNEGATIVE = 0.02
SMOOTH_MAX_BETA = 40.0

# Training control
RESUME_FROM_CHECKPOINT = True
SAVE_MODEL_STATE_DICTS = False  # keep False unless model files are explicitly needed

assert RUN_MODE in {"SMOKE", "FULL"}
print("RUN_MODE:", RUN_MODE)


In [ ]:
# ============================================================
# Cell 2 — Imports, deterministic runtime, paths, ZIP extraction
# ============================================================
from __future__ import annotations
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import sys, json, math, time, random, hashlib, shutil, zipfile, platform, traceback, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
except Exception as exc:
    raise ImportError(
        "PyTorch is required. Install it in the active Jupyter environment, restart the kernel, "
        "and run all cells again. Example: %pip install torch"
    ) from exc

from scipy.stats import spearmanr
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, ShuffleSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=RuntimeWarning)

SEED = RANDOM_STATE
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
torch.set_num_threads(max(1, int(TORCH_NUM_THREADS)))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _locate_repository_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for root in [candidate, *candidate.parents]:
        if (
            (root / "notebooks").is_dir()
            and (root / "data").is_dir()
            and (root / "results").is_dir()
            and (root / "provenance").is_dir()
        ):
            return root
    raise FileNotFoundError("Could not locate the clean-room repository root.")


REPOSITORY_ROOT = _locate_repository_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from software.cmt_repository_paths import artifact_namespace, runtime_cache_root

CWD = REPOSITORY_ROOT


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def safe_relpath(path) -> str:
    try:
        resolved = Path(path).resolve()
        return resolved.relative_to(REPOSITORY_ROOT).as_posix()
    except Exception:
        return str(path)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def write_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)


NB09_DIR = artifact_namespace("02", REPOSITORY_ROOT)
NB09B_DIR = artifact_namespace("03", REPOSITORY_ROOT)
NB10_DIR = artifact_namespace("04", REPOSITORY_ROOT)

BASE_DIR = artifact_namespace("05", REPOSITORY_ROOT)
PROCESSED_DIR = BASE_DIR / "processed"
METRICS_DIR = BASE_DIR / "metrics"
AUDIT_DIR = BASE_DIR / "audit"
METADATA_DIR = BASE_DIR / "metadata"
LOG_DIR = BASE_DIR / "logs"
CHECKPOINT_DIR = runtime_cache_root(REPOSITORY_ROOT) / "notebook_05" / "checkpoints"
MODEL_DIR = BASE_DIR / "models"

for d in [PROCESSED_DIR, METRICS_DIR, AUDIT_DIR, METADATA_DIR, LOG_DIR, CHECKPOINT_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EVENT_ROWS = []
def log_event(stage, level, message, extra=None):
    EVENT_ROWS.append({
        "timestamp_utc": utc_now(), "stage": stage, "level": level,
        "message": message, "extra_json": json.dumps(extra or {}, default=str),
    })
    if level in {"WARNING", "ERROR"}:
        print(f"[{level}] {stage}: {message}")

def save_event_log():
    pd.DataFrame(EVENT_ROWS).to_csv(LOG_DIR / "05_event_log.csv", index=False)

print("Notebook 05 initialized")
print("Device:", DEVICE)
print("Notebook 02:", safe_relpath(NB09_DIR))
print("Notebook 03:", safe_relpath(NB09B_DIR))
print("Notebook 04:", safe_relpath(NB10_DIR))
print("Outputs:", safe_relpath(BASE_DIR))
log_event("init", "INFO", "Notebook 05 initialized", {"run_mode": RUN_MODE, "device": str(DEVICE)})
save_event_log()


In [ ]:
# ============================================================
# Cell 3 — Input decisions, required files, and hashes
# ============================================================
REQUIRED = {
    "nb09_final": NB09_DIR / "metadata" / "02_final_decision.json",
    "nb09_features": NB09_DIR / "processed" / "02_master_feature_table.csv",
    "nb09_metadata_targets": NB09_DIR / "processed" / "02_master_metadata_and_targets.csv",
    "nb09_masks": NB09_DIR / "audit" / "02_target_plausibility_masks.csv",
    "nb09b_final": NB09B_DIR / "metadata" / "03_final_decision.json",
    "nb09b_feature_sets": NB09B_DIR / "processed" / "03_compiled_feature_sets_by_target.csv",
    "nb09b_matrix": NB09B_DIR / "processed" / "03_compiled_target_feature_matrix.csv",
    "nb10_final": NB10_DIR / "metadata" / "04_final_decision.json",
    "nb10_config": NB10_DIR / "metadata" / "04_benchmark_config.json",
    "nb10_split_audit": NB10_DIR / "audit" / "04_split_composition_audit.csv",
}
missing = [k for k,p in REQUIRED.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")

with open(REQUIRED["nb09_final"], encoding="utf-8") as f:
    d09 = json.load(f)
with open(REQUIRED["nb09b_final"], encoding="utf-8") as f:
    d09b = json.load(f)
with open(REQUIRED["nb10_final"], encoding="utf-8") as f:
    d10 = json.load(f)
with open(REQUIRED["nb10_config"], encoding="utf-8") as f:
    config10 = json.load(f)

assert str(d09.get("final_decision", "")).startswith("FULL_GO"), d09
assert d09b.get("final_decision") == "FULL_GO_TO_NOTEBOOK_10B", d09b
assert d10.get("final_decision") == "FULL_GO_TO_NOTEBOOK_11", d10

hash_rows=[]
for role,p in REQUIRED.items():
    hash_rows.append({"role":role, "relative_path":safe_relpath(p), "size_bytes":p.stat().st_size, "sha256":sha256_file(p)})
pd.DataFrame(hash_rows).to_csv(METADATA_DIR / "05_input_file_hashes.csv", index=False)

print("Input decisions accepted:")
print("  Notebook 02:", d09.get("final_decision"))
print("  Notebook 03:", d09b.get("final_decision"))
print("  Notebook 04:", d10.get("final_decision"))


In [ ]:
# ============================================================
# Cell 4 — Load authoritative Notebook 04-aligned dataset and compile common features
# ============================================================
TARGETS = [
    "average_voltage",
    "capacity_grav",
    "energy_grav",
    "max_delta_volume",
    "stability_charge",
    "stability_discharge",
    "stability_worst",
]
TARGET_UNITS = {
    "average_voltage": "V",
    "capacity_grav": "mAh/g",
    "energy_grav": "Wh/kg",
    "max_delta_volume": "database_volume_change",
    "stability_charge": "eV/atom",
    "stability_discharge": "eV/atom",
    "stability_worst": "eV/atom",
}
TARGET_INDEX = {t:i for i,t in enumerate(TARGETS)}
IDX_V = TARGET_INDEX["average_voltage"]
IDX_Q = TARGET_INDEX["capacity_grav"]
IDX_E = TARGET_INDEX["energy_grav"]
IDX_VOL = TARGET_INDEX["max_delta_volume"]
IDX_SC = TARGET_INDEX["stability_charge"]
IDX_SD = TARGET_INDEX["stability_discharge"]
IDX_SW = TARGET_INDEX["stability_worst"]

# Notebook 04 used 02_master_feature_table.csv directly as master_df and merged
# only the required mask columns. Reproduce that ordering here.
master_df = pd.read_csv(REQUIRED["nb09_features"], low_memory=False)
metadata_targets_df = pd.read_csv(REQUIRED["nb09_metadata_targets"], low_memory=False)
mask_df = pd.read_csv(REQUIRED["nb09_masks"], low_memory=False)
compiled_sets_df = pd.read_csv(REQUIRED["nb09b_feature_sets"])
compiled_matrix_df = pd.read_csv(REQUIRED["nb09b_matrix"])

assert master_df["record_index"].is_unique
assert metadata_targets_df["record_index"].is_unique
assert mask_df["record_index"].is_unique

required_id_cols = [
    "record_index", "working_ion", "electrode_uid", "framework_uid",
    "host_chemsys_no_working_ion", "coarse_family",
]
missing_id_cols = [c for c in required_id_cols if c not in master_df.columns]
if missing_id_cols:
    raise ValueError(f"Missing required identifier/group columns: {missing_id_cols}")

required_mask_cols = [
    "record_index",
    "mask_physics_plausible_all_targets",
    "mask_voltage_0_to_6",
    "mask_capacity_grav_positive_le_1000",
    "mask_energy_grav_nonnegative_le_6000",
    "mask_volume_change_0_to_2",
    "mask_stability_worst_0_to_2",
]
missing_mask_cols = [c for c in required_mask_cols if c not in mask_df.columns]
if missing_mask_cols:
    raise ValueError(f"Missing required mask columns: {missing_mask_cols}")

benchmark_df = master_df.merge(
    mask_df[required_mask_cols],
    on="record_index",
    how="left",
    validate="one_to_one",
)
for col in required_mask_cols:
    if col != "record_index":
        benchmark_df[col] = benchmark_df[col].fillna(False).astype(bool)

assert len(benchmark_df) == 3354
assert benchmark_df["record_index"].tolist() == master_df["record_index"].tolist()

# The linked-target model requires all seven linked outputs to be present.
valid_mask = benchmark_df["mask_physics_plausible_all_targets"].astype(bool)
valid_mask &= benchmark_df[TARGETS].apply(pd.to_numeric, errors="coerce").notna().all(axis=1)
VALID_INDICES = np.flatnonzero(valid_mask.to_numpy())
assert len(VALID_INDICES) == 3201, f"Expected 3201 valid records, found {len(VALID_INDICES)}"

PROTOCOLS = ["P2"] if RUN_MODE == "SMOKE" else list(FULL_PROTOCOLS)

feature_manifest_rows=[]
COMMON_FEATURES={}
for protocol in PROTOCOLS:
    sets=[]
    per_target_counts={}
    for target in TARGETS:
        s=set(compiled_sets_df.loc[
            (compiled_sets_df.target==target) & (compiled_sets_df.protocol==protocol),
            "feature",
        ].astype(str))
        if not s:
            raise RuntimeError(f"No compiler features for {target}/{protocol}")
        sets.append(s)
        per_target_counts[target]=len(s)
    common=sorted(set.intersection(*sets))
    common=[
        c for c in common
        if c in benchmark_df.columns and pd.api.types.is_numeric_dtype(benchmark_df[c])
    ]
    if not common:
        raise RuntimeError(f"No common numeric features for protocol {protocol}")
    COMMON_FEATURES[protocol]=common
    for order,feature in enumerate(common, start=1):
        feature_manifest_rows.append({
            "protocol":protocol,
            "feature_order":order,
            "feature":feature,
            "common_safe_for_targets":"|".join(TARGETS),
        })
    print(protocol, "common safe features:", len(common), "per-target counts:", per_target_counts)

feature_manifest_df=pd.DataFrame(feature_manifest_rows)
feature_manifest_df.to_csv(
    PROCESSED_DIR / "05_common_multitask_feature_manifest.csv",
    index=False,
)

# Explicit exclusion audit: no target column may enter X.
audit_rows=[]
for protocol, features in COMMON_FEATURES.items():
    for target in TARGETS:
        audit_rows.append({
            "protocol":protocol,
            "target":target,
            "target_present_as_feature":target in features,
            "n_common_features":len(features),
            "pass":target not in features,
        })
target_input_audit_df=pd.DataFrame(audit_rows)
target_input_audit_df.to_csv(
    AUDIT_DIR / "05_target_input_exclusion_audit.csv",
    index=False,
)
assert target_input_audit_df["pass"].all()

pd.DataFrame({
    "record_index": benchmark_df.iloc[VALID_INDICES]["record_index"].to_numpy(),
}).to_csv(PROCESSED_DIR / "05_valid_record_universe.csv", index=False)

print("Valid linked-target rows:", len(VALID_INDICES))


In [ ]:
# ============================================================
# Cell 5 — Reconstruct Notebook 04 validation design and persist record-level folds
# ============================================================
RANDOM_N_SPLITS = int(config10["random_n_splits"])
GROUP_KFOLD_SPLITS = int(config10["group_kfold_splits"])
LEAVE_CHEMSYS_MAX_GROUPS = int(config10["leave_chemsys_max_groups"])
MIN_TEST_RECORDS_FOR_LEAVEOUT = int(config10["min_test_records_for_leaveout"])


def make_random_splits(valid_indices):
    splitter=ShuffleSplit(
        n_splits=RANDOM_N_SPLITS,
        test_size=0.20,
        random_state=RANDOM_STATE,
    )
    dummy=np.zeros((len(valid_indices),1))
    out=[]
    for fold_id,(tr,te) in enumerate(splitter.split(dummy)):
        out.append({
            "split_name":"random_split",
            "fold_id":fold_id,
            "heldout_group":f"random_{fold_id}",
            "train_idx":valid_indices[tr],
            "test_idx":valid_indices[te],
            "construction":"ShuffleSplit_same_seed_and_row_universe",
        })
    return out


def make_framework_splits(valid_indices):
    groups=benchmark_df.iloc[valid_indices]["framework_uid"].astype(str).fillna("missing_group").to_numpy()
    n_eff=min(GROUP_KFOLD_SPLITS,len(pd.unique(groups)))
    splitter=GroupKFold(n_splits=n_eff)
    dummy=np.zeros((len(valid_indices),1))
    out=[]
    for fold_id,(tr,te) in enumerate(splitter.split(dummy,groups=groups)):
        heldout_groups=sorted(set(groups[te]))
        out.append({
            "split_name":"framework_groupkfold",
            "fold_id":fold_id,
            "heldout_group":"|".join(heldout_groups[:20]),
            "train_idx":valid_indices[tr],
            "test_idx":valid_indices[te],
            "construction":"GroupKFold_same_group_field_and_fold_count",
        })
    return out


# Notebook 04 saved the held-out group identity for every leave-one-group-out fold.
# Reconstruct those folds directly from the saved reference table, eliminating
# pandas value_counts tie-order sensitivity.
ref=pd.read_csv(REQUIRED["nb10_split_audit"])
ref=ref[ref.target=="average_voltage"].copy()
ref["fold_id"]=pd.to_numeric(ref["fold_id"], errors="raise").astype(int)
ref["n_train"]=pd.to_numeric(ref["n_train"], errors="raise").astype(int)
ref["n_test"]=pd.to_numeric(ref["n_test"], errors="raise").astype(int)


def make_reference_leave_splits(valid_indices, column, split_name, max_folds=None):
    reference=ref.loc[ref.split_name==split_name].sort_values("fold_id").copy()
    if max_folds is not None:
        reference=reference.head(int(max_folds))
    groups=benchmark_df.iloc[valid_indices][column].astype(str).fillna("missing_group").to_numpy()
    out=[]
    for row in reference.itertuples(index=False):
        group=str(row.heldout_group)
        is_test=(groups==group)
        te=np.flatnonzero(is_test)
        tr=np.flatnonzero(~is_test)
        if len(te)==0:
            raise RuntimeError(
                f"Reference held-out group {group!r} for {split_name} is absent from the current valid universe"
            )
        out.append({
            "split_name":split_name,
            "fold_id":int(row.fold_id),
            "heldout_group":group,
            "train_idx":valid_indices[tr],
            "test_idx":valid_indices[te],
            "construction":"reference_heldout_group_reconstruction",
        })
    return out


all_splits=[]
all_splits += make_random_splits(VALID_INDICES)
all_splits += make_framework_splits(VALID_INDICES)
all_splits += make_reference_leave_splits(
    VALID_INDICES, "coarse_family", "leave_family_out"
)
chemsys_folds = LEAVE_CHEMSYS_MAX_GROUPS if INCLUDE_ALL_75_CHEMICAL_SYSTEM_HOLDOUTS else min(20, LEAVE_CHEMSYS_MAX_GROUPS)
all_splits += make_reference_leave_splits(
    VALID_INDICES,
    "host_chemsys_no_working_ion",
    "leave_chemical_system_out",
    max_folds=chemsys_folds,
)
all_splits += make_reference_leave_splits(
    VALID_INDICES, "working_ion", "leave_working_ion_out"
)

if RUN_MODE=="SMOKE":
    selected_splits=[
        s for s in all_splits
        if s["split_name"]=="random_split" and s["fold_id"]==0
    ]
else:
    selected_splits=[
        s for s in all_splits
        if s["split_name"] in FULL_SPLITS
    ]

# Compatibility audit:
# - leave-one-group-out folds require exact held-out group identity and counts;
# - random folds require the same seed/design and counts;
# - framework folds require the same grouping design and fold sizes.
# Notebook 04 did not save record-level framework assignments, and GroupKFold
# tie handling can vary across NumPy/scikit-learn versions. We record, but do
# not falsely require, the truncated held-out-group string to be identical.
ref_key={
    (r.split_name,int(r.fold_id)):
    (str(r.heldout_group),int(r.n_train),int(r.n_test))
    for r in ref.itertuples()
}
audit=[]
for s in selected_splits:
    key=(s["split_name"],int(s["fold_id"]))
    expected=ref_key.get(key)
    observed=(str(s["heldout_group"]),len(s["train_idx"]),len(s["test_idx"]))
    reference_found=expected is not None
    heldout_match=reference_found and expected[0]==observed[0]
    n_train_match=reference_found and expected[1]==observed[1]
    n_test_match=reference_found and expected[2]==observed[2]

    if s["split_name"] in {
        "leave_family_out",
        "leave_chemical_system_out",
        "leave_working_ion_out",
    }:
        compatibility_level="exact_heldout_group_and_counts"
        strict_pass=reference_found and heldout_match and n_train_match and n_test_match
        record_level_exact_reuse_verified=True
    elif s["split_name"]=="random_split":
        compatibility_level="same_shuffle_design_seed_universe_and_counts"
        strict_pass=reference_found and heldout_match and n_train_match and n_test_match
        record_level_exact_reuse_verified=False
    elif s["split_name"]=="framework_groupkfold":
        compatibility_level="same_groupkfold_design_group_field_and_fold_sizes"
        strict_pass=reference_found and n_train_match and n_test_match
        record_level_exact_reuse_verified=False
    else:
        compatibility_level="unknown"
        strict_pass=False
        record_level_exact_reuse_verified=False

    audit.append({
        "split_name":key[0],
        "fold_id":key[1],
        "heldout_group":observed[0],
        "n_train":observed[1],
        "n_test":observed[2],
        "construction":s["construction"],
        "reference_found":reference_found,
        "heldout_group_match":heldout_match,
        "n_train_match":n_train_match,
        "n_test_match":n_test_match,
        "record_level_exact_reuse_verified":record_level_exact_reuse_verified,
        "compatibility_level":compatibility_level,
        "pass":bool(strict_pass),
    })

split_audit_df=pd.DataFrame(audit)
split_audit_df.to_csv(
    AUDIT_DIR / "05_split_design_compatibility_audit.csv",
    index=False,
)
# Retain the original expected filename for downstream compatibility.
split_audit_df.to_csv(
    AUDIT_DIR / "05_split_reuse_audit.csv",
    index=False,
)

if not split_audit_df["pass"].all():
    display(split_audit_df[~split_audit_df["pass"]].head(20))
    raise RuntimeError("Notebook 04 split-design compatibility audit failed")

framework_nonidentical=split_audit_df.loc[
    (split_audit_df.split_name=="framework_groupkfold")
    & (~split_audit_df.heldout_group_match)
]
if len(framework_nonidentical):
    print(
        "Framework fold note:",
        len(framework_nonidentical),
        "folds have version-sensitive group assignments but matching GroupKFold design and fold sizes.",
    )

split_plan_df=pd.DataFrame([{
    "split_name":s["split_name"],
    "fold_id":s["fold_id"],
    "heldout_group":s["heldout_group"],
    "n_train":len(s["train_idx"]),
    "n_test":len(s["test_idx"]),
    "construction":s["construction"],
} for s in selected_splits])
split_plan_df.to_csv(PROCESSED_DIR / "05_split_plan.csv", index=False)

# Persist exact Notebook 05 test membership so every later method and notebook
# can reuse the same records without regenerating version-sensitive folds.
record_manifest_rows=[]
for s in selected_splits:
    test_records=benchmark_df.iloc[np.asarray(s["test_idx"],dtype=int)]["record_index"].to_numpy()
    for record_index in test_records:
        record_manifest_rows.append({
            "split_name":s["split_name"],
            "fold_id":int(s["fold_id"]),
            "heldout_group":s["heldout_group"],
            "record_index":record_index,
            "set_role":"test",
        })
record_split_manifest_df=pd.DataFrame(record_manifest_rows)
record_split_manifest_df.to_csv(
    PROCESSED_DIR / "05_outer_test_record_manifest.csv",
    index=False,
)

print("Selected outer folds:",len(selected_splits))
print(split_plan_df.groupby("split_name").size())
print("Persisted test-membership rows:",len(record_split_manifest_df))


In [ ]:
# ============================================================
# Cell 6 — Multi-task network, fold-only preprocessing, physics loss
# ============================================================
class MultiTaskMLP(nn.Module):
    def __init__(self, n_features, n_outputs, hidden_dims=(64,32)):
        super().__init__()
        layers=[]
        prev=n_features
        for width in hidden_dims:
            layers += [nn.Linear(prev,width), nn.ReLU(), nn.LayerNorm(width)]
            prev=width
        layers.append(nn.Linear(prev,n_outputs))
        self.net=nn.Sequential(*layers)
    def forward(self,x):
        return self.net(x)


def split_train_inner(train_indices, fraction, seed):
    rng=np.random.default_rng(seed)
    idx=np.asarray(train_indices,dtype=int).copy()
    rng.shuffle(idx)
    n_val=max(32,int(round(len(idx)*fraction)))
    n_val=min(n_val,max(1,len(idx)-64))
    return idx[n_val:], idx[:n_val]


def fit_preprocessors(train_idx, feature_names):
    Xraw=benchmark_df.loc[train_idx,feature_names].apply(pd.to_numeric,errors="coerce").to_numpy(dtype=float)
    Yraw=benchmark_df.loc[train_idx,TARGETS].apply(pd.to_numeric,errors="coerce").to_numpy(dtype=float)
    imputer=SimpleImputer(strategy="median")
    scaler_x=StandardScaler()
    scaler_y=StandardScaler()
    X=scaler_x.fit_transform(imputer.fit_transform(Xraw))
    Y=scaler_y.fit_transform(Yraw)
    return imputer,scaler_x,scaler_y,X,Y


def transform_xy(indices, feature_names, imputer, scaler_x, scaler_y):
    Xraw=benchmark_df.loc[indices,feature_names].apply(pd.to_numeric,errors="coerce").to_numpy(dtype=float)
    Yraw=benchmark_df.loc[indices,TARGETS].apply(pd.to_numeric,errors="coerce").to_numpy(dtype=float)
    X=scaler_x.transform(imputer.transform(Xraw))
    Y=scaler_y.transform(Yraw)
    return X,Y,Yraw


def smooth_max(a,b,beta=40.0):
    stacked=torch.stack([a,b],dim=-1)
    return torch.logsumexp(beta*stacked,dim=-1)/beta


def physics_components(pred_std, y_mean_t, y_scale_t):
    pred=pred_std*y_scale_t+y_mean_t
    v,q,e=pred[:,IDX_V],pred[:,IDX_Q],pred[:,IDX_E]
    sc,sd,sw=pred[:,IDX_SC],pred[:,IDX_SD],pred[:,IDX_SW]
    energy_res=(e-v*q)/torch.clamp(y_scale_t[IDX_E],min=1e-8)
    sw_smooth=smooth_max(sc,sd,beta=SMOOTH_MAX_BETA)
    stability_res=(sw-sw_smooth)/torch.clamp(y_scale_t[IDX_SW],min=1e-8)
    nonnegative_cols=[IDX_V,IDX_Q,IDX_E,IDX_VOL,IDX_SC,IDX_SD,IDX_SW]
    negative=[]
    for j in nonnegative_cols:
        negative.append(torch.relu(-pred[:,j]/torch.clamp(y_scale_t[j],min=1e-8)))
    nonneg=torch.stack(negative,dim=1)
    return energy_res,stability_res,nonneg,pred


def train_model(X_train,Y_train,X_val,Y_val,y_mean,y_scale, constrained, seed, max_epochs, patience):
    torch.manual_seed(seed)
    model=MultiTaskMLP(X_train.shape[1],Y_train.shape[1],HIDDEN_DIMS).to(DEVICE)
    optimizer=torch.optim.Adam(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    mse=nn.MSELoss()
    train_ds=TensorDataset(torch.tensor(X_train,dtype=torch.float32),torch.tensor(Y_train,dtype=torch.float32))
    generator=torch.Generator().manual_seed(seed)
    loader=DataLoader(train_ds,batch_size=min(BATCH_SIZE,len(train_ds)),shuffle=True,generator=generator)
    Xv=torch.tensor(X_val,dtype=torch.float32,device=DEVICE)
    Yv=torch.tensor(Y_val,dtype=torch.float32,device=DEVICE)
    ym=torch.tensor(y_mean,dtype=torch.float32,device=DEVICE)
    ys=torch.tensor(y_scale,dtype=torch.float32,device=DEVICE)
    best_state=None; best_val=float("inf"); bad=0; history=[]
    for epoch in range(max_epochs):
        model.train(); losses=[]
        for xb,yb in loader:
            xb=xb.to(DEVICE); yb=yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            pred=model(xb)
            data_loss=mse(pred,yb)
            er,sr,neg,_=physics_components(pred,ym,ys)
            physics_loss=(er.pow(2).mean()+sr.pow(2).mean())
            nonneg_loss=neg.pow(2).mean()
            loss=data_loss
            if constrained:
                loss=loss+LAMBDA_ENERGY*er.pow(2).mean()+LAMBDA_STABILITY*sr.pow(2).mean()+LAMBDA_NONNEGATIVE*nonneg_loss
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        model.eval()
        with torch.no_grad():
            pv=model(Xv)
            val_data=float(mse(pv,Yv).cpu())
            er,sr,neg,_=physics_components(pv,ym,ys)
            val_phys=float((er.pow(2).mean()+sr.pow(2).mean()).cpu())
            val_score=val_data + (0.15*val_phys if constrained else 0.0)
        history.append({"epoch":epoch,"train_loss":float(np.mean(losses)),"val_data_loss":val_data,"val_physics_loss":val_phys,"val_score":val_score})
        if val_score < best_val-1e-6:
            best_val=val_score
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            bad=0
        else:
            bad+=1
            if bad>=patience:
                break
    if best_state is None:
        raise RuntimeError("Training produced no valid model state")
    model.load_state_dict(best_state)
    return model,pd.DataFrame(history),best_val


def predict_physical(model,X,scaler_y):
    model.eval()
    with torch.no_grad():
        pred_std=model(torch.tensor(X,dtype=torch.float32,device=DEVICE)).cpu().numpy()
    return scaler_y.inverse_transform(pred_std)

print("Model and physics-loss functions ready")


In [ ]:
# ============================================================
# Cell 7 — Metrics and prediction variants
# ============================================================
def safe_spearman(y_true,y_pred):
    if len(y_true)<3 or np.nanstd(y_true)==0 or np.nanstd(y_pred)==0:
        return np.nan
    return float(spearmanr(y_true,y_pred,nan_policy="omit").statistic)


def metric_row(y_true,y_pred):
    return {
        "rmse":float(np.sqrt(mean_squared_error(y_true,y_pred))),
        "mae":float(mean_absolute_error(y_true,y_pred)),
        "r2":float(r2_score(y_true,y_pred)) if len(y_true)>=2 and np.std(y_true)>0 else np.nan,
        "spearman":safe_spearman(y_true,y_pred),
    }


def make_variants(pred_unconstrained,pred_constrained):
    u_direct=pred_unconstrained.copy()
    u_derived=pred_unconstrained.copy()
    u_derived[:,IDX_E]=u_derived[:,IDX_V]*u_derived[:,IDX_Q]
    u_derived[:,IDX_SW]=np.maximum(u_derived[:,IDX_SC],u_derived[:,IDX_SD])
    c_direct=pred_constrained.copy()
    c_derived=pred_constrained.copy()
    c_derived[:,IDX_E]=c_derived[:,IDX_V]*c_derived[:,IDX_Q]
    c_derived[:,IDX_SW]=np.maximum(c_derived[:,IDX_SC],c_derived[:,IDX_SD])
    return {
        "unconstrained_direct":u_direct,
        "unconstrained_hardderived":u_derived,
        "softconstrained_direct":c_direct,
        "softconstrained_hardderived":c_derived,
    }


def consistency_summary(pred):
    energy_res=pred[:,IDX_E]-pred[:,IDX_V]*pred[:,IDX_Q]
    stability_res=pred[:,IDX_SW]-np.maximum(pred[:,IDX_SC],pred[:,IDX_SD])
    negative_count=(pred<0).sum(axis=1)
    return {
        "energy_consistency_mae":float(np.mean(np.abs(energy_res))),
        "energy_consistency_rmse":float(np.sqrt(np.mean(energy_res**2))),
        "stability_consistency_mae":float(np.mean(np.abs(stability_res))),
        "stability_consistency_rmse":float(np.sqrt(np.mean(stability_res**2))),
        "any_negative_prediction_rate":float(np.mean(negative_count>0)),
        "mean_negative_outputs_per_record":float(np.mean(negative_count)),
    }


In [ ]:
# ============================================================
# Cell 8 — Training-only constraint sensitivity diagnostic
# ============================================================
# This diagnostic never uses the outer test fold and does not alter the locked lambdas.
sensitivity_rows=[]
reference_split=selected_splits[0]
reference_protocol=PROTOCOLS[0]
features=COMMON_FEATURES[reference_protocol]
outer_train=np.asarray(reference_split["train_idx"],dtype=int)
fit_idx,val_idx=split_train_inner(outer_train,INNER_VALIDATION_FRACTION,RANDOM_STATE+700)
imputer,sx,sy,Xfit,Yfit=fit_preprocessors(fit_idx,features)
Xval,Yval,_=transform_xy(val_idx,features,imputer,sx,sy)

orig=(LAMBDA_ENERGY,LAMBDA_STABILITY)
for lam in [0.0,0.1,0.3,1.0,3.0]:
    # The training function reads globals. Change temporarily for a training-only diagnostic.
    globals()["LAMBDA_ENERGY"]=lam
    globals()["LAMBDA_STABILITY"]=lam
    model,hist,best=train_model(Xfit,Yfit,Xval,Yval,sy.mean_,sy.scale_,constrained=(lam>0),seed=RANDOM_STATE+int(lam*100)+1,max_epochs=SMOKE_MAX_EPOCHS,patience=SMOKE_EARLY_STOPPING_PATIENCE)
    pred=predict_physical(model,Xval,sy)
    cs=consistency_summary(pred)
    row={"lambda":lam,"protocol":reference_protocol,"split_name":reference_split["split_name"],"fold_id":reference_split["fold_id"],"best_validation_score":best}
    row.update(cs); sensitivity_rows.append(row)

globals()["LAMBDA_ENERGY"],globals()["LAMBDA_STABILITY"]=orig
sensitivity_df=pd.DataFrame(sensitivity_rows)
sensitivity_df.to_csv(METRICS_DIR / "05_constraint_strength_sensitivity.csv",index=False)
display(sensitivity_df)
print("Locked full-run lambdas restored:",LAMBDA_ENERGY,LAMBDA_STABILITY)


In [ ]:
# ============================================================
# Cell 9 — Main matched outer-validation execution with checkpoint/resume
# ============================================================
PREDICTION_PATH=CHECKPOINT_DIR / "05_predictions_checkpoint.csv"
METRIC_PATH=CHECKPOINT_DIR / "05_fold_metrics_checkpoint.csv"
CONSISTENCY_PATH=CHECKPOINT_DIR / "05_consistency_checkpoint.csv"
EXECUTION_PATH=CHECKPOINT_DIR / "05_execution_checkpoint.csv"

existing_keys=set()
if RESUME_FROM_CHECKPOINT and EXECUTION_PATH.exists():
    ex=pd.read_csv(EXECUTION_PATH)
    ok=ex[ex.status=="OK"] if "status" in ex.columns else ex
    existing_keys=set(zip(ok.protocol.astype(str),ok.split_name.astype(str),ok.fold_id.astype(int)))
    print("Resume keys found:",len(existing_keys))


def append_df(df,path):
    if df is None or len(df)==0:
        return
    path.parent.mkdir(parents=True,exist_ok=True)
    df.to_csv(path,mode="a",header=not path.exists(),index=False)

execution_rows=[]
start_all=time.time()
max_epochs=SMOKE_MAX_EPOCHS if RUN_MODE=="SMOKE" else FULL_MAX_EPOCHS
patience=SMOKE_EARLY_STOPPING_PATIENCE if RUN_MODE=="SMOKE" else FULL_EARLY_STOPPING_PATIENCE

for protocol in PROTOCOLS:
    features=COMMON_FEATURES[protocol]
    for split in selected_splits:
        key=(protocol,split["split_name"],int(split["fold_id"]))
        if key in existing_keys:
            continue
        t0=time.time(); status="OK"; error=""
        try:
            outer_train=np.asarray(split["train_idx"],dtype=int)
            outer_test=np.asarray(split["test_idx"],dtype=int)
            fit_idx,val_idx=split_train_inner(outer_train,INNER_VALIDATION_FRACTION,RANDOM_STATE+1000+int(split["fold_id"]))

            # Fit every preprocessing object only on the model-fit part of the outer training fold.
            imputer,sx,sy,Xfit,Yfit=fit_preprocessors(fit_idx,features)
            Xval,Yval,_=transform_xy(val_idx,features,imputer,sx,sy)
            Xtest,Ytest_std,Ytest=transform_xy(outer_test,features,imputer,sx,sy)

            u_model,u_hist,u_best=train_model(
                Xfit,Yfit,Xval,Yval,sy.mean_,sy.scale_,False,
                RANDOM_STATE+2000+int(split["fold_id"]),max_epochs,patience,
            )
            c_model,c_hist,c_best=train_model(
                Xfit,Yfit,Xval,Yval,sy.mean_,sy.scale_,True,
                RANDOM_STATE+3000+int(split["fold_id"]),max_epochs,patience,
            )
            pred_u=predict_physical(u_model,Xtest,sy)
            pred_c=predict_physical(c_model,Xtest,sy)
            variants=make_variants(pred_u,pred_c)

            # Wide predictions: one row per outer-test record and method variant.
            pred_rows=[]; metric_rows=[]; consistency_rows=[]
            rec=benchmark_df.loc[outer_test,["record_index","working_ion","framework_uid","coarse_family","host_chemsys_no_working_ion"]].reset_index(drop=True)
            for variant,pred in variants.items():
                out=rec.copy()
                out.insert(0,"protocol",protocol)
                out.insert(1,"split_name",split["split_name"])
                out.insert(2,"fold_id",int(split["fold_id"]))
                out.insert(3,"heldout_group",split["heldout_group"])
                out.insert(4,"model_variant",variant)
                for j,target in enumerate(TARGETS):
                    out[f"true_{target}"]=Ytest[:,j]
                    out[f"pred_{target}"]=pred[:,j]
                    mr={
                        "protocol":protocol,"split_name":split["split_name"],"fold_id":int(split["fold_id"]),
                        "heldout_group":split["heldout_group"],"model_variant":variant,
                        "target":target,"target_unit":TARGET_UNITS[target],"n_train":len(outer_train),"n_test":len(outer_test),
                        "n_features":len(features),
                    }
                    mr.update(metric_row(Ytest[:,j],pred[:,j])); metric_rows.append(mr)
                cs={
                    "protocol":protocol,"split_name":split["split_name"],"fold_id":int(split["fold_id"]),
                    "heldout_group":split["heldout_group"],"model_variant":variant,"n_test":len(outer_test),
                }
                cs.update(consistency_summary(pred)); consistency_rows.append(cs)
                pred_rows.append(out)

            append_df(pd.concat(pred_rows,ignore_index=True),PREDICTION_PATH)
            append_df(pd.DataFrame(metric_rows),METRIC_PATH)
            append_df(pd.DataFrame(consistency_rows),CONSISTENCY_PATH)

            if SAVE_MODEL_STATE_DICTS:
                torch.save({"state_dict":u_model.state_dict(),"features":features,"targets":TARGETS},MODEL_DIR/f"05_{protocol}_{split['split_name']}_{split['fold_id']}_unconstrained.pt")
                torch.save({"state_dict":c_model.state_dict(),"features":features,"targets":TARGETS},MODEL_DIR/f"05_{protocol}_{split['split_name']}_{split['fold_id']}_constrained.pt")

        except Exception as exc:
            status="ERROR"; error=f"{type(exc).__name__}: {exc}"
            log_event("training","ERROR",error,{"protocol":protocol,"split_name":split["split_name"],"fold_id":split["fold_id"],"traceback":traceback.format_exc()[-3000:]})
        execution_rows.append({
            "protocol":protocol,"split_name":split["split_name"],"fold_id":int(split["fold_id"]),
            "heldout_group":split["heldout_group"],"status":status,"error":error,
            "n_train":len(split["train_idx"]),"n_test":len(split["test_idx"]),"n_features":len(features),
            "elapsed_seconds":time.time()-t0,"timestamp_utc":utc_now(),
        })
        append_df(pd.DataFrame([execution_rows[-1]]),EXECUTION_PATH)
        save_event_log()
        print(f"{status}: {protocol} | {split['split_name']} | fold {split['fold_id']} | {time.time()-t0:.1f}s")

print("Main execution elapsed minutes:",(time.time()-start_all)/60)


In [ ]:
# ============================================================
# Cell 10 — Finalize canonical outputs, aggregate metrics, audits
# ============================================================
execution_df=pd.read_csv(EXECUTION_PATH)
metric_df=pd.read_csv(METRIC_PATH) if METRIC_PATH.exists() else pd.DataFrame()
consistency_df=pd.read_csv(CONSISTENCY_PATH) if CONSISTENCY_PATH.exists() else pd.DataFrame()
prediction_df=pd.read_csv(PREDICTION_PATH) if PREDICTION_PATH.exists() else pd.DataFrame()

# Deduplicate safely after resume.
execution_df=execution_df.drop_duplicates(["protocol","split_name","fold_id"],keep="last")
metric_df=metric_df.drop_duplicates(["protocol","split_name","fold_id","model_variant","target"],keep="last")
consistency_df=consistency_df.drop_duplicates(["protocol","split_name","fold_id","model_variant"],keep="last")
prediction_df=prediction_df.drop_duplicates(["protocol","split_name","fold_id","model_variant","record_index"],keep="last")

execution_df.to_csv(AUDIT_DIR / "05_config_execution_audit.csv",index=False)
metric_df.to_csv(METRICS_DIR / "05_independent_vs_constrained_fold_metrics.csv",index=False)
consistency_df.to_csv(METRICS_DIR / "05_physics_consistency_metrics.csv",index=False)
prediction_df.to_csv(PROCESSED_DIR / "05_physics_constrained_oof_predictions.csv",index=False)

# Aggregate across folds; do not use unweighted group R² as the sole interpretation.
def agg_ci(g,col):
    x=pd.to_numeric(g[col],errors="coerce").dropna().to_numpy()
    if len(x)==0: return pd.Series({f"{col}_mean":np.nan,f"{col}_sd":np.nan,f"{col}_ci95_low":np.nan,f"{col}_ci95_high":np.nan})
    mean=float(np.mean(x)); sd=float(np.std(x,ddof=1)) if len(x)>1 else 0.0
    half=1.96*sd/math.sqrt(len(x)) if len(x)>1 else 0.0
    return pd.Series({f"{col}_mean":mean,f"{col}_sd":sd,f"{col}_ci95_low":mean-half,f"{col}_ci95_high":mean+half})

base_keys=["protocol","split_name","model_variant","target","target_unit"]
summary=metric_df.groupby(base_keys,dropna=False).agg(
    n_folds=("fold_id","nunique"),n_test_min=("n_test","min"),n_test_median=("n_test","median"),
    n_features=("n_features","median"),
).reset_index()
for col in ["rmse","mae","r2","spearman"]:
    rows=[]
    for keys,g in metric_df.groupby(base_keys,dropna=False):
        row=dict(zip(base_keys,keys if isinstance(keys,tuple) else (keys,)))
        row.update(agg_ci(g,col).to_dict())
        rows.append(row)
    tmp=pd.DataFrame(rows)
    summary=summary.merge(tmp,on=base_keys,how="left")
summary.to_csv(METRICS_DIR / "05_independent_vs_hardderived_vs_softconstrained_metrics.csv",index=False)

# Paired method differences at the fold level for later Notebook 06.
pair_keys=["protocol","split_name","fold_id","target"]
pivot=metric_df.pivot_table(index=pair_keys,columns="model_variant",values=["mae","rmse","r2","spearman"]).reset_index()
pivot.columns=["_".join([str(x) for x in c if str(x)!=""]) if isinstance(c,tuple) else str(c) for c in pivot.columns]
pivot.to_csv(METRICS_DIR / "05_paired_method_comparisons.csv",index=False)

# Relationship-specific slim tables.
energy_cols=[c for c in prediction_df.columns if c in ["protocol","split_name","fold_id","heldout_group","model_variant","record_index","working_ion","true_average_voltage","pred_average_voltage","true_capacity_grav","pred_capacity_grav","true_energy_grav","pred_energy_grav"]]
prediction_df[energy_cols].to_csv(PROCESSED_DIR / "05_energy_direct_vs_derived_predictions.csv",index=False)
stability_cols=[c for c in prediction_df.columns if c in ["protocol","split_name","fold_id","heldout_group","model_variant","record_index","working_ion","true_stability_charge","pred_stability_charge","true_stability_discharge","pred_stability_discharge","true_stability_worst","pred_stability_worst"]]
prediction_df[stability_cols].to_csv(PROCESSED_DIR / "05_stability_direct_vs_derived_predictions.csv",index=False)

# Preprocessing-scope audit is construction-based and tied to every completed fold.
preprocess_audit=execution_df.copy()
preprocess_audit["imputer_fit_scope"]="inner_fit_subset_of_outer_train_only"
preprocess_audit["x_scaler_fit_scope"]="inner_fit_subset_of_outer_train_only"
preprocess_audit["y_scaler_fit_scope"]="inner_fit_subset_of_outer_train_only"
preprocess_audit["outer_test_used_for_fit_or_early_stopping"]=False
preprocess_audit["pass"]=preprocess_audit.status.eq("OK")
preprocess_audit.to_csv(AUDIT_DIR / "05_preprocessing_scope_audit.csv",index=False)

print("Completed configurations:",execution_df.status.value_counts().to_dict())
print("Prediction rows:",len(prediction_df))


In [ ]:
# ============================================================
# Cell 11 — Go/no-go gates, manifests, final decision
# ============================================================
expected_configs=len(PROTOCOLS)*len(selected_splits)
ok_configs=int((execution_df.status=="OK").sum())
error_configs=int((execution_df.status!="OK").sum())

# Required method variants and targets.
required_variants={"unconstrained_direct","unconstrained_hardderived","softconstrained_direct","softconstrained_hardderived"}
observed_variants=set(metric_df.model_variant.astype(str)) if len(metric_df) else set()
observed_targets=set(metric_df.target.astype(str)) if len(metric_df) else set()

# Physics value gate: soft constrained should reduce at least one consistency MAE relative to unconstrained direct.
cons_agg=consistency_df.groupby("model_variant")[["energy_consistency_mae","stability_consistency_mae"]].mean() if len(consistency_df) else pd.DataFrame()
physics_value=False
if {"unconstrained_direct","softconstrained_direct"}.issubset(cons_agg.index):
    physics_value=(
        cons_agg.loc["softconstrained_direct","energy_consistency_mae"] < cons_agg.loc["unconstrained_direct","energy_consistency_mae"]
        or cons_agg.loc["softconstrained_direct","stability_consistency_mae"] < cons_agg.loc["unconstrained_direct","stability_consistency_mae"]
    )

# The final full-run gate requires complete execution. Smoke mode has its own controlled decision.
gates=[
    ("input_decisions_accepted",True),
    ("compiler_false_safe_zero",int(d09b.get("false_safe_n",-1))==0),
    ("target_columns_excluded_from_features",bool(target_input_audit_df["pass"].all())),
    ("notebook10_split_design_compatible",bool(split_audit_df["pass"].all())),
    ("all_expected_configurations_completed",ok_configs==expected_configs and error_configs==0),
    ("all_method_variants_present",required_variants.issubset(observed_variants)),
    ("all_linked_targets_present",set(TARGETS).issubset(observed_targets)),
    ("preprocessing_train_only",bool(preprocess_audit["pass"].all()) if len(preprocess_audit) else False),
    ("physics_constraint_reduces_a_consistency_error",bool(physics_value)),
]
gate_df=pd.DataFrame(gates,columns=["gate","pass"])
gate_df.to_csv(AUDIT_DIR / "05_go_no_go_gate_audit.csv",index=False)
display(gate_df)

if RUN_MODE=="SMOKE":
    required_smoke=["input_decisions_accepted","compiler_false_safe_zero","target_columns_excluded_from_features","notebook10_split_design_compatible","all_expected_configurations_completed","all_method_variants_present","all_linked_targets_present","preprocessing_train_only"]
    smoke_pass=gate_df.set_index("gate").loc[required_smoke,"pass"].all()
    final_decision="SMOKE_TEST_PASS" if smoke_pass else "HOLD_SMOKE_TEST_FAILURE"
    next_step="Set RUN_MODE='FULL', restart kernel, and run all cells."
else:
    full_pass=gate_df["pass"].all()
    final_decision="FULL_GO_TO_NOTEBOOK_10C" if full_pass else "HOLD_PHYSICS_CONSTRAINED_VALIDATION_FAILURE"
    next_step="Proceed to Notebook 06 only after independent audit of this complete output package."

# Environment and policy metadata.
env={
    "run_timestamp_utc":utc_now(),"python":sys.version.split()[0],"platform":platform.platform(),
    "numpy":np.__version__,"pandas":pd.__version__,"torch":torch.__version__,"device":str(DEVICE),
    "run_mode":RUN_MODE,"random_state":RANDOM_STATE,
}
write_json(env,METADATA_DIR / "05_software_environment.json")
write_json({
    "final_decision":final_decision,"run_timestamp_utc":utc_now(),"run_mode":RUN_MODE,
    "expected_configurations":expected_configs,"ok_configurations":ok_configs,"error_configurations":error_configs,
    "physics_value_gate":bool(physics_value),"next_step":next_step,
    "no_dft_performed":True,"no_candidate_ranking_performed":True,"no_manuscript_writing_performed":True,
    "notebook10_record_level_split_manifest_available":False,
    "notebook10b_record_level_split_manifest_persisted":True,
    "split_reuse_claim":"leave-out groups reconstructed exactly; random/framework design-compatible; all Notebook 05 methods share one exact persisted manifest",
},METADATA_DIR / "05_final_decision.json")

# Output manifest with relative paths only.
manifest=[]
for p in sorted(BASE_DIR.rglob("*")):
    if p.is_file() and p.name!="05_output_file_manifest.csv":
        manifest.append({"relative_path":p.relative_to(BASE_DIR).as_posix(),"size_bytes":p.stat().st_size,"sha256":sha256_file(p)})
pd.DataFrame(manifest).to_csv(METADATA_DIR / "05_output_file_manifest.csv",index=False)

save_event_log()
print("\nFINAL DECISION:",final_decision)
print(next_step)


## Expected decisions

### First run

With `RUN_MODE = "SMOKE"`, the notebook should end with:

```text
FINAL DECISION: SMOKE_TEST_PASS
```

### Final run

After changing `RUN_MODE = "FULL"`, restarting the kernel, and running all cells, the notebook should end with:

```text
FINAL DECISION: FULL_GO_TO_NOTEBOOK_10C
```

Do not proceed to Notebook 06 when the final decision begins with `HOLD_`.
